# Save the directions for every model in GAAS table

### Global vals and helper function set up

In [1]:
import os
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.manifold import TSNE

from robustbench import load_model
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# CIFAR-10 class names
CLASSES = ["airplane","automobile","bird","cat","deer",
           "dog","frog","horse","ship","truck"]

MODELS = [
            "Bartoldson2024Adversarial_WRN-94-16", 
            "Amini2024MeanSparse_S-WRN-94-16",
            "Peng2023Robust",
            "Bai2024MixedNUTS",
            "Sehwag2021Proxy_ResNest152",
            "Gowal2021Improving_R18_ddpm_100m",
            "Debenedetti2022Light_XCiT-L12",
            "Sehwag2021Proxy_R18"
        ]

CAT_IDX = 3
DOG_IDX = 5
BINARY_CLASSES = {CAT_IDX: "cat", DOG_IDX: "dog"}

# L-inf threat-model budget (8/255 is the standard CIFAR-10 benchmark)
EPSILON = 8 / 255

print(f"Binary subset: {CAT_IDX}=cat  {DOG_IDX}=dog  epsilon={EPSILON:.5f}")

def pkl_save(a, path):
    with open(path, 'wb') as handle:
        pickle.dump(a, handle, protocol=pickle.HIGHEST_PROTOCOL)

def pkl_load(path):
    with open(path, 'rb') as handle:
        return pickle.load(handle)

/home/kwest/um/S2P/MRP-Sem_2-Group_8/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Binary subset: 3=cat  5=dog  epsilon=0.03137


/home/kwest/um/S2P/MRP-Sem_2-Group_8/.venv/lib/python3.10/site-packages/robustbench/loaders.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Model Loading

1. Bartoldson2024Adversarial_WRN-94-16
2. Amini2024MeanSparse_S-WRN-94-16
3. Peng2023Robust
4. Bai2024MixedNUTS
5. Sehwag2021Proxy_ResNest152
6. Gowal2021Improving_R18_ddpm_100m
7. Debenedetti2022Light_XCiT-L12
8. Sehwag2021Proxy_R18

In [ ]:
# I'm gonna not load all of these at once, instead load when you need
loaded_model = load_model(model_name=MODELS[0],
                     dataset="cifar10",
                     threat_model="Linf").to(device)
loaded_model.eval()

# Load all cat/dog data

In [3]:
# load n data pts, 2000 = all cat/dog pts
def build_catdog_dataset(n=2000):
    """Sample n Cat and Dog images from the CIFAR-10 test set."""
    ds         = datasets.CIFAR10(root="./data", train=False,
                                   download=True, transform=transforms.ToTensor())
    indices    = [i for i, (_, lbl) in enumerate(ds)
                  if lbl in (CAT_IDX, DOG_IDX)]
    np.random.seed(42)
    chosen     = np.random.choice(indices, size=min(n, len(indices)), replace=False)
    loader     = DataLoader(Subset(ds, chosen),
                            batch_size=len(chosen), shuffle=False)
    imgs, lbls = next(iter(loader))
    return imgs, lbls

print("\nFalling back to live CIFAR-10 sampling ...")
ci, cl       = build_catdog_dataset()
clean_images = ci.to(device)
clean_labels = cl.to(device)
print(f"Sampled {len(clean_labels)} Cat/Dog images from test set.")


Falling back to live CIFAR-10 sampling ...
Sampled 2000 Cat/Dog images from test set.


In [4]:
# just double checking we got 1000 of both classes
print(clean_labels[clean_labels == CAT_IDX].shape)
print(clean_labels[clean_labels == DOG_IDX].shape)
print(clean_images.shape)

torch.Size([1000])
torch.Size([1000])
torch.Size([2000, 3, 32, 32])


# Setup functions for evaluating model on only cat/dog

In [20]:
def pred_model(m, clean_images):
    # returns the predicted label of either cat or dog
    with torch.no_grad():
        out = m(clean_images)

    pred = out[:, [CAT_IDX, DOG_IDX]].argmax(dim=1) # 0 = cat, 1 = dog
    pred[pred == 0] = CAT_IDX
    pred[pred == 1] = DOG_IDX

    return pred

In [ ]:
pred = pred_model(loaded_model, clean_images)

torch.Size([2000])

In [ ]:

acc_a = (pred == clean_labels).float().mean().item()

print(f"Model A clean accuracy (cat/dog only): {acc_a:.1%}")
print()
print(f"{'Img':<5} {'True':<6} {'Pred A':<12} {'Pred B':<12}")
print("-" * 37)
for i in range(len(clean_labels)):
    t  = CLASSES[clean_labels[i].item()]
    pa = CLASSES[pred[i].item()]
    print(f"{i:<5} {t:<6} {'ok ' if pa==t else 'WRONG '}{pa:<6}")